## Tree-Based Models Comparison

In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
from src.data_utils import load_processed
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from src.model import save_model
import joblib

In [3]:
df=load_processed('model_ready.csv')

### Train-Test Split

In [4]:
X=df.drop(columns=['TARGET'])
y=df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    stratify=y, 
                                                    test_size=0.2, 
                                                    random_state=42)

### XGBoost Model

In [5]:
cat_col = df.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_col)
    ],
    remainder='passthrough'
)

pipe_xgb = Pipeline([
    ('prep', preprocessor),
    ('model', XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc"
    ))
])

In [6]:
pipe_xgb.fit(X_train, y_train)

y_pred_xgb = pipe_xgb.predict_proba(X_test)[:,1]
y_train_xgb = pipe_xgb.predict_proba(X_train)[:,1]

xgbv_auc = roc_auc_score(y_test, y_pred_xgb)
xgbt_auc = roc_auc_score(y_train, y_train_xgb)

print(f'XGBoost Train AUC {xgbt_auc}')
print(f'XGBoost Validation AUC {xgbv_auc}')

XGBoost Train AUC 0.7825213577172261
XGBoost Validation AUC 0.7726232518641848


#### Results

Train AUC: 0.7825    
Validation AUC: 0.7726    


In [75]:
save_model(pipe_xgb, 'xgb_v1_auc_0_7726.joblib')

✅ Model saved at: C:\Users\ASUS\OneDrive\Desktop\AI\RiskForge\models\xgb_v1_auc_0_7726.joblib


### LightGBM Model

In [6]:
pipe_lgbm = Pipeline([
    ('prep', preprocessor),
    ('model', LGBMClassifier(
        n_estimators=600,
        learning_rate=0.03,
        num_leaves=21,
        min_child_samples=80,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42
    ))
])

In [7]:
pipe_lgbm.fit(X_train, y_train)

y_pred_lgbm = pipe_lgbm.predict_proba(X_test)[:,1]
y_train_lgbm = pipe_lgbm.predict_proba(X_train)[:,1]

lgbmv_auc = roc_auc_score(y_test, y_pred_lgbm)
lgbmt_auc = roc_auc_score(y_train, y_train_lgbm)

print(f'LightGBM Train AUC {lgbmt_auc}')
print(f'LightGBM Validation AUC {lgbmv_auc}')

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.174019 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19337
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 258
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
LightGBM Train AUC 0.8198459840779081
LightGBM Validation AUC 0.7775344315653703


#### Results

Train AUC: 0.8198    
Validation AUC: 0.7775   


In [74]:
save_model(pipe_lgbm, 'lgbm_v1_auc_0_777.joblib')

✅ Model saved at: C:\Users\ASUS\OneDrive\Desktop\AI\RiskForge\models\lgbm_v1_auc_0_777.joblib


In [12]:
save_model({
    "model": pipe_lgbm,
    "X_test": X_test,
    "y_test": y_test
}, "bundle.joblib")

✅ Model saved at: C:\Users\ASUS\OneDrive\Desktop\AI\RiskForge\models\bundle.joblib


### Final Model Comparison (ROC-AUC)

| Model                | Train ROC-AUC   | Validation ROC-AUC   | Overfitting Gap   |
|----------------------|-----------------|----------------------|-------------------|
| Logistic Regression  | 0.7572          | 0.7586               | -0.0014           |
| Decision Tree        | 0.7286          | 0.7119               | 0.0167            |
| Random Forest        | 0.7819          | 0.7459               | 0.0360            |
| Gradient Boosting    | 0.7713          | 0.7654               | 0.0059            |
| XGBoost              | 0.7825          | 0.7726               | 0.0099            |
| LightGBM             | 0.8198          | 0.7775               | 0.0423            |

### Conclusion

LightGBM achieved the highest validation ROC-AUC (0.7775) with minimal overfitting (gap = 0.0423).    
Random Forest showed the largest overfitting gap, while Logistic Regression appeared slightly underfitting.     
Overall, LightGBM provides best balance between performance and generalization.    